# Data Gathering Pipeline: Indian Marriage Laws

This notebook handles the automated data acquisition phase of the project using a **Dual-Source Architecture** to overcome the limitations of Indian government websites.

### Methodology:
1. **Directory Sourcing (India Code):** We scrape `indiacode.nic.in` to acquire the definitive, official list of all marriage-related Acts, Rules, and Notifications.
2. **Content Sourcing (Indian Kanoon):** Because India Code often traps legal text in scanned PDFs, we programmatically query `indiankanoon.org` using our official list to extract the raw, scrapeable HTML text for each law. For some legal acts, we referenced the scanned PDFs from the India Code as well and added the data manually.
3. **Parsing & Chunking:** We parse the Indian Kanoon HTML to extract metadata and segment the legal text by section for our Vector Database.

#### **Setup and Dependencies**
Importing necessary libraries and setting up the local directory structure for data storage.

In [ ]:
# !pip install pandas lxml rapidfuzz beautifulsoup4 requests

import sys
import pathlib

# Add project root to path so 'config' package is importable from notebooks/
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import pandas as pd
from bs4 import BeautifulSoup
import requests
from rapidfuzz import fuzz
from urllib.parse import urljoin, quote
from concurrent.futures import ThreadPoolExecutor, as_completed
import json

from config import config

# Ensure the output directories exist (paths are defined in config.py).
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
config.LAWS_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Formatting options for pandas DataFrames
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

#### **Discovering Relevant Laws (via India Code)**
We scrape the official India Code portal to compile a master list of all acts, rules, and notifications related to the keyword "marriage".

In [ ]:
# Define the URL to scrape
url = "https://www.indiacode.nic.in/handle/123456789/1362/simple-search?query=marriage&searchradio=all"

# Read and parse the HTML content
html_content = requests.get(url, headers=HEADERS).text
soup = BeautifulSoup(html_content, 'html.parser')

# Find the div containing the relevant legal document links
all_div = soup.find('div', id='all')

laws = {}
for a_tag in soup.find_all('a', href=True):
    # Filter for standard legal classes
    valid_classes = ['allacts', 'allrules', 'allcircular', 'allnotification']
    if any(cls in a_tag.get('class', []) for cls in valid_classes):
        law_name = a_tag.get_text(strip=True)
        href = a_tag['href'].strip()
        
        # Avoid duplicates, keeping the first occurrence
        if law_name not in laws:
            laws[law_name] = href

# Create DataFrame and save initial master list
df = pd.DataFrame(list(laws.items()), columns=['Law Name', 'Href'])
df.to_csv(config.LAWS_CSV, index=False)
display(df.head())

In [3]:
df

,Law Name,Href
0,THE HARYANA COMPULSORY REGISTRATION OF MARRIAG...,/ViewSelectedActDetailsServlet?act_name=THE HA...
1,THE ODISHA MUHAMMEDAN MARRIAGES AND DIVORCES R...,/ViewSelectedActDetailsServlet?act_name=THE OD...
2,"The Parsi Marriage and Divorce Act, 1936",/ViewSelectedActDetailsServlet?act_name=The Pa...
3,"The Foreign Marriage Act, 1969",/ViewSelectedActDetailsServlet?act_name=The Fo...
4,"The Anand Marriage Act, 1909",/ViewSelectedActDetailsServlet?act_name=The An...
5,"The Tripura Recording of Marriage Act, 2003",/ViewSelectedActDetailsServlet?act_name=The Tr...
6,"The Dissolution of Muslim Marriages Act, 1939",/ViewSelectedActDetailsServlet?act_name=The Di...
7,"The Hindu Marriage Act, 1955",/ViewSelectedActDetailsServlet?act_name=The Hi...
8,"The Prohibition of Child Marriage Act, 2006",/ViewSelectedActDetailsServlet?act_name=The Pr...
9,The Uttarakhand Compulsory Registration of Mar...,/ViewSelectedActDetailsServlet?act_name=The Ut...


#### **Generating Indian Kanoon Search Queries**
Now that we have the official names from India Code, we map them to search queries for the Indian Kanoon database to fetch the actual text.

In [4]:
page_urls = {}

for law in df['Law Name'].tolist():
    query = quote(law)
    page_url = f"https://indiankanoon.org/search/?formInput={query}"
    page_urls[law] = page_url
    
print(f"Generated {len(page_urls)} search queries.")

Generated 76 search queries.


#### **Extracting Exact Document Links (Concurrent Scraping)**
We use concurrent threading to quickly search Indian Kanoon. Since search results aren't always perfect, we use `rapidfuzz` to score the search results against our official India Code law name, selecting the best match.

In [ ]:
BASE_URL = "https://indiankanoon.org"

def process_law(law_name, url):
    """
    Fetches search results from Indian Kanoon, fuzzy matches the law name 
    against the result titles, and extracts the link to the 'entire act' document.
    
    Args:
        law_name (str): The name of the law to search for.
        url (str): The Indian Kanoon search URL.
        
    Returns:
        dict: A dictionary containing the law name, the best matched URL, and the match score.
              Returns None if an error occurs.
    """
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        best_match = None
        best_score = -1

        law_lower = law_name.lower()
        main_term = law_lower.split(",")[0]

        # Iterate through search results to find the best fuzzy match
        for article in soup.select("article.result"):
            title_tag = article.select_one("h4.result_title a")
            if not title_tag:
                continue

            title = title_tag.get_text(" ", strip=True)
            title_lower = title.lower()

            score = fuzz.token_set_ratio(law_lower, title_lower)
            if main_term in title_lower:
                score += 20  # Boost score if primary term is present

            if score > best_score:
                best_score = score
                best_match = article

        entire_act_link = None
        if best_match:
            # Look specifically for the 'entire act' link on the result card
            tag = best_match.find("a", string=lambda x: x and "entire" in x.lower())
            if tag:
                href = tag.get("href")
                entire_act_link = urljoin(BASE_URL, href)

        return {
            "law": law_name,
            "link": entire_act_link,
            "score": best_score
        }
    except Exception as e:
        print(f"Error processing {law_name}: {e}")
        return None

# Execute the scraping concurrently
results = []
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(process_law, law, url): law for law, url in page_urls.items()}
    
    for future in as_completed(futures):
        result = future.result()
        if result:
            results.append(result)

raw_links_df = pd.DataFrame(results)
raw_links_df.to_csv(config.LAWS_WITH_LINKS_CSV, index=False)
display(raw_links_df.head())

#### **Step 4: HTML Parsing & Data Structuring**
This function defines the extraction logic to separate the raw HTML gathered from Indian Kanoon into semantic chunks (sections) required for the Retrieval-Augmented Generation (RAG) vector database.

In [6]:
def parse_indian_kanoon_act(html_content):
    """
    Parses the HTML content of an Indian Kanoon act page to extract metadata
    and chunk the document by legal sections.
    
    Args:
        html_content (str): The raw HTML content of the act page.
        
    Returns:
        dict: A structured dictionary containing 'document_metadata' and a list of 'chunks'.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # 1. Extract Global Metadata
    metadata = {
        "document_title": soup.find('h2', class_='doc_title').get_text(strip=True) if soup.find('h2', class_='doc_title') else "Unknown",
        "act_number": soup.find('h2', string=lambda t: t and 'Act' in t).get_text(strip=True) if soup.find('h2', string=lambda t: t and 'Act' in t) else "Unknown",
        "jurisdiction": "India",
        "document_type": "Act"
    }
    
    # Extract date from publication info
    pub_info = soup.find('li', class_='publication-info')
    if pub_info:
        metadata["enactment_date"] = pub_info.get_text(strip=True).replace("Published in Gazette 25 on ", "")

    chunks = []
    
    # 2. Extract Sections (Chunks)
    sections = soup.find_all('section', class_='akn-section')
    
    for sec in sections:
        header = sec.find('h3')
        if not header:
            continue
            
        full_title = header.get_text(strip=True)
        parts = full_title.split('.', 1)
        sec_number = parts[0].strip() if len(parts) > 1 else "Unknown"
        sec_title = parts[1].strip() if len(parts) > 1 else full_title
        
        # Extract content: Remove the header so it doesn't duplicate in the body text
        header.extract() 
        
        # Get text, replace multiple spaces/newlines with single spaces
        content = ' '.join(sec.get_text(separator=' ', strip=True).split())
        
        if content and content != "***":
            chunks.append({
                "act_number": metadata["act_number"],
                "section_number": sec_number,
                "section_title": sec_title,
                "content": content
            })
            
    return {
        "document_metadata": metadata,
        "chunks": chunks
    }

#### **Step 5: Final Data Extraction and JSON Export**
*(Note: `final_law_with_links.csv` represents the manually verified and cleaned version of the links extracted in Step 3).* 

We iterate through our verified Indian Kanoon URLs, fetch the full HTML, pass it to our parser, and export clean JSON representations to the `/data` directory for embedding into Qdrant later.

In [ ]:
# Load the manually verified dataset
final_df = pd.read_csv(config.FINAL_LAW_LINKS_CSV)
session = requests.Session()
session.headers.update(HEADERS)

for index, row in final_df.iterrows():
    # Skip empty links
    if pd.isna(row['link']):
        continue
        
    # Sanitize file name by replacing spaces and invalid characters
    safe_name = "".join([c for c in row['law'] if c.isalpha() or c.isdigit() or c==' ']).rstrip()
    file_name = f"{safe_name.replace(' ', '_')}.json"
    full_file_path = config.LAWS_DIR / file_name

    # Avoid redundant processing if file exists
    if full_file_path.exists():
        print(f"[SKIPPED] File already exists for: {row['law']}")
        continue

    try:
        response = session.get(row['link'])
        response.raise_for_status() 
        
        parsed_data = parse_indian_kanoon_act(response.text)

        with open(full_file_path, 'w', encoding='utf-8') as f:
            json.dump(parsed_data, f, ensure_ascii=False, indent=2)
            
        print(f"[SUCCESS] Data saved for: {row['law']}")

    except Exception as e:
        print(f"[ERROR] Failed to download {row['law']}: {e}")